<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_supplement_helmholtz_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 Supplement to PDE Recap (Optional) — Helmholtz Equation: the Stator Slot

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.2 · Fundamental PDEs**

*Optional.* This notebook stands on its own, beside the set's silicon die and
vibrating panel, and is not part of the report. It takes L7.2's **Helmholtz**
equation to a real device: the stator slot of an electrical machine, where hot
copper turns Poisson's equation into Helmholtz's. Everything it needs is in
this folder - you do not need to have done Ex_07.1, which uses the same slot.

### The slot

A 10 × 20 mm slot in the stator of an electrical machine, its walls held at the
cooled iron's 90 °C, holding 32 round copper wires of 2 mm that all carry the
same current. The bundle conducts heat at $k_{\text{eff}} = 0.70$ W/m·K. The
temperature rise $\theta = T - 90$ °C obeys $k_{\text{eff}}\nabla^2\theta + q = 0$
with $\theta = 0$ on the walls, and the copper's heat $q$ grows with its
temperature.

![One stator slot, with half of each neighbour](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/Ex07.2_slot.png)

### Sections

| | what you do | TODO |
|---|---|---|
| **1** | why hot copper turns Poisson into Helmholtz | — |
| **2** | the slot's first eigenvalue, and how far it pushes the temperature | — |
| **3** | a network finds the eigenvalue, with and without its safeguard | the loss |
| **4** | three levers a designer can pull | — |
| **5** | two questions | your answers |

About a minute on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

**What these cells do.** Fetch the library files, import them, and describe the slot.

**How.** `course_core.py` and `pinn_core.py` are this set's; `slot_problem.py` is the slot: its geometry, its heat and a finite-difference solver. It is a copy of the module Ex_07.1 uses, kept in this folder so this notebook stands on its own.

In [ ]:
# files-cell (supplement) -------------------------------------------------
# Locally the files are copied from the repository; on Google Colab they are
# fetched from the public course repository on every run, so a file left over
# from an earlier session cannot shadow an updated one. Run this cell first.
import os, sys, shutil, urllib.request
BASE = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/"
FILES = {"course_core.py": "Ex07.2-heat-and-wave/course_core.py",
         "pinn_core.py": "Ex07.2-heat-and-wave/pinn_core.py",
         "slot_problem.py": "Ex07.2-heat-and-wave/slot_problem.py"}   # the stator slot
ON_COLAB = "google.colab" in sys.modules
for f, src in FILES.items():
    if not ON_COLAB and os.path.exists(f):
        continue
    local = os.path.join("..", src)
    if not ON_COLAB and os.path.exists(local):
        shutil.copy(local, f)
    else:
        urllib.request.urlretrieve(BASE + src, f)
    print("fetched", f)
for _m in ("course_core", "pinn_core", "slot_problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
import time
from pinn_core import *                                  # noqa: F401,F403
import slot_problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

set_seed(88)  # fix the random start so a run repeats
print("device:", DEVICE, " dtype:", torch.get_default_dtype())
pb.describe_slot()

---

## 1 · Why Poisson becomes Helmholtz

**What this cell does.** Solves the slot at rated current (31.4 A) twice — once with the copper's resistance frozen at the wall temperature, once as it really is — and prints the difference.

**How.** The physics is four steps:

1. At rated current the copper in the middle of the slot sits about 19 K above the wall.
2. Copper's resistance rises about 0.3 % per kelvin at these temperatures, so it is about 6 % higher there.
3. The same current through a higher resistance makes more heat, $RI^2$.
4. More heat means the slot must be a little hotter before its walls can carry it all away. It settles a little higher.

That is **positive feedback**: more heat, hotter copper, more resistance, more heat. It settles because it is weak. In the equation it is one term. Split the heat into its fixed part and the part that grows with the rise, $q = c_0 + c_1\theta$:

$$k_{\text{eff}}\,\nabla^2\theta + c_0 + \underbrace{c_1\,\theta}_{\text{feedback}} = 0
\qquad\Longrightarrow\qquad
\nabla^2\theta + \frac{c_1}{k_{\text{eff}}}\,\theta = -\frac{c_0}{k_{\text{eff}}}$$

Without the feedback, $c_1 = 0$, this is **Poisson** — L7.1's equation. With it, the left side is **Helmholtz**, $\nabla^2 u + \lambda u$ from L7.2, with $\lambda = c_1/k_{\text{eff}}$. Nothing here is a wave: heat spreads and smooths, and the Helmholtz *operator* is all the two share.

In [ ]:
A0, _ = pb.fdm_matrix(0.0, 81)                               # conduction only: k_eff x the Laplacian
c0, c1 = pb.heat_coefficients(pb.I_RATED)                    # heat = c0 + c1 * rise
frozen = spla.spsolve(A0, -c0 * np.ones(A0.shape[0])).max()  # resistance kept at the wall's
hot = pb.fdm_solve(pb.I_RATED, 81)[2].max()                  # resistance as it really is

more_R = pb.resistivity(pb.T_WALL + hot) / pb.resistivity(pb.T_WALL) - 1
print(f"copper at the hot spot: {hot:.2f} K above the wall, resistance {more_R:.1%} higher")
print(f"rise, resistance frozen at 90 C : {frozen:6.2f} K")
print(f"rise, hot copper                : {hot:6.2f} K   ({hot / frozen - 1:.1%} more)")

**What you should see.** Resistance about 6 % higher at the hot spot, and a
rise about 4.5 % higher than with the resistance frozen — less than the
resistance's rise, because only the middle of the slot is that hot; near the
walls the copper is barely warmer than the iron.

---

## 2 · The eigenvalue, and how far it pushes the temperature

**What this cell does.** Finds the slot's first eigenvalue two ways, then solves the slot at forty currents up to it and plots how much hot copper pushes the temperature up.

**How.** L7.2's Helmholtz problem is

$$\nabla^2 u + \lambda u = 0 \ \text{ in the slot}, \qquad u = 0 \ \text{ on the walls},$$

and it has non-zero solutions only for special values of $\lambda$, the eigenvalues. The smallest, $\lambda_1$, is a property of the slot's shape alone — **how easily it sheds heat to its walls** — and for a $W \times H$ rectangle there is a formula, $\lambda_1 = \pi^2(1/W^2 + 1/H^2)$. The cell also takes it from the finite-difference matrix, as the smallest eigenvalue of the grid's Laplacian.

The feedback is then one ratio, $c_1/(k_{\text{eff}}\,\lambda_1)$: how strong the feedback is, over how easily the slot sheds heat. The temperature is pushed up by a factor of about $1/(1 - \text{ratio})$. Because $c_1$ grows as $I^2$, the ratio is $(I/I^*)^2$ — and at $I^*$, where it reaches 1, there is no steady temperature at all: **thermal runaway**. The dashed lines mark where the hot spot would reach the insulation's thermal classes, which is where a real machine is limited.

In [ ]:
from scipy.optimize import brentq

lam1 = np.pi ** 2 * (1 / (2 * pb.A_HALF) ** 2 + 1 / (2 * pb.B_HALF) ** 2)   # the formula, 1/m^2
lam1_fdm = spla.eigsh(-A0 / pb.K_EFF, k=1, sigma=0, which="LM")[0][0]      # the grid's own
I_star = pb.runaway_current()                                                # where c1 / k_eff = lambda_1
print(f"lambda_1: {lam1:.2e} 1/m^2 from the formula, {lam1_fdm:.2e} from the grid "
      f"({abs(lam1_fdm / lam1 - 1):.2%} apart)")
print(f"feedback ratio at rated current: {(pb.I_RATED / I_star)**2:.1%};  it reaches 100 % at {I_star:.0f} A")

currents = np.linspace(5, 0.97 * I_star, 40)
push = []
for I in currents:
    A, rhs = pb.fdm_matrix(I, 81)                          # with the feedback
    push.append(spla.spsolve(A, rhs).max() / spla.spsolve(A0, rhs).max())

classes = {"B": 130, "F": 155, "H": 180}                   # insulation thermal classes, C
limit = {c: brentq(lambda I: pb.fdm_solve(I, 41)[2].max() - (T - pb.T_WALL), 10, 150)
         for c, T in classes.items()}

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(currents, push, "o", ms=4, color=CYCLE[0], label="finite differences")
ax.plot(currents, 1 / (1 - (currents / I_star) ** 2), color="0.3", lw=1.5,
        label="$1/(1 - (I/I^*)^2)$, from the eigenvalue")
for c, I in limit.items():
    ax.axvline(I, color=CYCLE[2], ls="--", lw=1)
    ax.text(I, 14, f" class {c}", color=CYCLE[2], rotation=90, va="top")
ax.axvline(I_star, color="k", lw=1); ax.text(I_star, 14, f" {I_star:.0f} A", rotation=90, va="top")
ax.set_ylim(1, 15); ax.set_xlabel("current per wire  [A]")
ax.set_ylabel("rise with hot copper  /  rise frozen")
ax.set_title("how much hot copper pushes the temperature up")
ax.legend(frameon=False, loc="center left"); plt.show()
print("the insulation classes are reached at " +
      ", ".join(f"{c}: {I:.0f} A" for c, I in limit.items()))

**What you should see.** The two values of $\lambda_1$ 0.01 % apart,
and a curve that stays near 1 — a few per cent — across the currents a machine
uses, then climbs without limit as the current approaches 158 A. The
finite-difference points sit a little above the eigenvalue's curve, because the
slot's higher modes add a small amount the first one does not.

**The model's arithmetic, not the machine's.** Past the class lines the numbers
describe no real winding. Insulation is rated by thermal class: above its class
temperature it does not fail at once, it ages faster — roughly half the life for
every 10 K — and protection disconnects the machine long before. The eigenvalue
matters in the working range, where it says how big the push is.

---

## 3 · A network finds the eigenvalue

**What these cells do.** Build a network for the Helmholtz problem, write its loss, and train it twice — with and without one of its safeguards.

**How.** The Helmholtz problem has a trap, the one on L7.2's **Trivial Solution** slide: $u \equiv 0$ satisfies $\nabla^2 u + \lambda u = 0$ for *every* $\lambda$, with a residual of exactly zero, and gradient descent finds it first. Two safeguards exclude it:

* **a unit norm** — a loss term asking for $\overline{u^2} = 1$, so the mode cannot shrink to nothing;
* **a floor on $\lambda$** — $\lambda$ is trained through a softplus that keeps it above a chosen value.

The walls are built in with the mask of L7.1's **Hard Enforcement** slide, $u = (1 - \xi^2)(1 - \eta^2)\,\mathcal{N}(\xi, \eta)$, in the coordinates $\xi = x/a$ and $\eta = y/b$, where $a = 5$ mm and $b = 10$ mm are the slot's half-width and half-depth, so both run from −1 to 1. In them, the residual multiplied by $a^2$ is $u_{\xi\xi} + \tfrac14 u_{\eta\eta} + \Lambda u$, with $\Lambda = a^2\lambda$. The exact answer is $\Lambda_1 = \pi^2(1/4 + 1/16) = 3.08$.

In [ ]:
FLOOR = 1.0                                        # safeguard 2: Lambda is kept above this

class Mode(torch.nn.Module):                      # the mode u = D * N, and its eigenvalue Lambda
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=3)            # tanh network, 2 inputs
        self.lam_raw = torch.nn.Parameter(torch.tensor(0.0))       # trained beside the weights

    def lam(self):
        return FLOOR + torch.nn.functional.softplus(self.lam_raw)  # never below the floor

    def forward(self, p):
        D = (1 - p[:, :1] ** 2) * (1 - p[:, 1:] ** 2)              # the mask: zero on the walls
        return D * self.net(p)                                     # the network's output, masked

box = ((-1, 1), (-1, 1))                          # the slot in xi, eta
pts = to_tensor(interior_points(2000, box, method="lhs", seed=1), requires_grad=True)  # fixed for L-BFGS

In [ ]:
def loss_terms(model, p):
    u = model(p)                                                    # the mode at the points
    res = d2(u, p, 0) + 0.25 * d2(u, p, 1) + model.lam() * u       # a^2 (lap u + lambda u), by autograd
    norm = (u ** 2).mean() - 1.0                                    # safeguard 1: u may not vanish
    return res.pow(2).mean(), norm ** 2

In [ ]:
runs = {}
for name, w_norm, adam, lbfgs in (("with the norm", 10.0, 1500, 100),
                                  ("without it", 0.0, 800, 0)):
    set_seed(88)                                               # the same start for both
    model = Mode().to(DEVICE)                                  # a fresh network and Lambda

    def loss():                                                # what train_two_stage minimises
        r, n = loss_terms(model, pts)
        return r + w_norm * n                                  # w_norm = 0 drops safeguard 1

    t0 = time.perf_counter()
    train_two_stage(model, loss, adam_steps=adam, lbfgs_steps=lbfgs, lr=2e-3, report_every=0)  # Adam, then L-BFGS
    with torch.no_grad():                                      # evaluate only, no graph
        msq = model(pts).pow(2).mean().item()
    runs[name] = (model, model.lam().item(), msq, time.perf_counter() - t0)

EXACT = np.pi ** 2 * (1 / 4 + 1 / 16)
for name, (model, lam, msq, dt) in runs.items():
    print(f"{name:14s}: Lambda {lam:.2f} (exact {EXACT:.2f}, error {abs(lam / EXACT - 1):.2%}),"
          f"  mean u^2 {msq:.1e},  {dt:.0f} s")

lam_pinn = runs["with the norm"][1] / pb.A_HALF ** 2           # back to 1/m^2
print(f"\nthermal-runaway current from the network's eigenvalue: {I_star * np.sqrt(lam_pinn / lam1):.0f} A"
      f"  (from the formula: {I_star:.0f} A)")

X, Y, P = grid_points(41, 81, pb.DOMAIN)
xi_eta = to_tensor(P / np.array([pb.A_HALF, pb.B_HALF]))       # the grid, scaled to -1..1
with torch.no_grad():                                          # evaluate only, no graph
    u = to_numpy(runs["with the norm"][0](xi_eta)).ravel()
exact = 2 * np.cos(np.pi * P[:, 0] / (2 * pb.A_HALF)) * np.cos(np.pi * P[:, 1] / (2 * pb.B_HALF))
u = u * np.sign(u @ exact)                                     # a mode has no preferred sign
fig, axes = plt.subplots(1, 2, figsize=(8.4, 5.4))
pb.plot_field(u, 41, 81, ax=axes[0], title="the network's mode", label="u", cmap="viridis")
pb.plot_field(u - exact, 41, 81, ax=axes[1], title="minus the exact mode", label="error",
              cmap="coolwarm")
plt.tight_layout(); plt.show()

**What you should see.** With the norm, $\Lambda$ = 3.08 against the exact
3.08, and a thermal-runaway current of 158 A, the same as the formula's;
the mode is a single dome, $\cos(\pi\xi/2)\cos(\pi\eta/2)$, and the error map
stays small against its peak of 2. **Without the norm, $\overline{u^2}$ falls
to almost nothing** — the network has found $u \equiv 0$, a perfect zero of the
residual that says nothing about the slot. That is the trap, and one loss term
is what closes it.

For a rectangle the formula already knew $\lambda_1$, which is what let you
score the network exactly. A real slot — a rounded bottom, tooth tips, a liner
— has no formula, and then a numerical eigenvalue is the only way to get it.

---

## 4 · Three levers a designer can pull

**What this cell does.** Works out the feedback ratio at 45 A for the slot as it is and for three changes a designer could make, and how each moves the hot spot.

**How.** The ratio is

$$\frac{c_1/k_{\text{eff}}}{\lambda_1} = \frac{\text{fill}\cdot\rho_{20}\,\alpha\,J^2}{k_{\text{eff}}\;\pi^2\,(1/W^2 + 1/H^2)},$$

so there are three levers: a lower current density $J$ (it is squared), a higher $k_{\text{eff}}$, and a larger $\lambda_1$ — a narrower slot. The hot spot itself scales roughly as $J^2/(k_{\text{eff}}\lambda_1)$, so the same levers lower it too. Copper's $\alpha$ is fixed by the material. And cooling the iron harder lowers the hot spot but **not** the ratio: $c_1$ is the resistance's rise *per kelvin*, the same at 60 °C as at 90 °C.

In [ ]:
def feedback(J, k=pb.K_EFF, W=2 * pb.A_HALF, H=2 * pb.B_HALF):
    lam = np.pi ** 2 * (1 / W ** 2 + 1 / H ** 2)                  # how easily the slot sheds heat
    return pb.FILL * pb.RHO_CU * pb.ALPHA_CU * J ** 2 / (k * lam), J ** 2 / (k * lam)

J45 = 45.0 / pb.A_WIRE                                            # A/m^2 at 45 A
base_ratio, base_hot = feedback(J45)
print(f"{'':32s}{'feedback ratio':>16s}{'hot spot vs now':>18s}")
for name, kw in (("the slot as it is, 45 A", dict(J=J45)),
                 ("current density halved", dict(J=J45 / 2)),
                 ("k_eff doubled (impregnation)", dict(J=J45, k=2 * pb.K_EFF)),
                 ("slot 8 mm wide, not 10", dict(J=J45, W=8e-3))):
    r, hot = feedback(**kw)
    print(f"  {name:30s}{r:15.1%}{hot / base_hot:17.0%}")

**What you should see.** About 8 % for the slot as it is, falling to 2 % with the current density halved, 4 % with $k_{\text{eff}}$ doubled and about 6 % with a narrower slot — and the hot spot falling with each, to a quarter, a half and about 70 % of what it is now.

These are the levers machine designers pull to raise a rating: more copper per
ampere, better impregnation, slots shaped for a short heat path. They are
pulled to lower the hot spot, not to avoid thermal runaway — a real winding
sits at a few per cent of it — but the eigenvalue says, for any of them, how
much of the hot spot is the copper heating itself.

---

## 5 · Before you move on

Answer these here; this notebook is not part of the report. Each builds part of an answer to one of L7.2's questions for the oral examination; the arrow says which.

1. Without the norm term the network returned $u \equiv 0$. Why is that a perfect solution of the residual, and what does each safeguard exclude?
   *→ L7.2 Q3*
2. What does $\lambda_1$ say about the slot, and why does the temperature run away as $c_1/k_{\text{eff}}$ approaches it?
   *→ L7.2 Q5*

*Your answers:*

1.
2.